# LAB | Extractive Question Answering

<div style="border:1px solid #ccc; border-radius:6px; text-align: center;">

<br>

**Run this notebook in [Google Colab](https://colab.research.google.com/) for GPU acceleration.**

<br>

</div>

This notebook demonstrates how Pinecone helps you build an extractive question-answering application. To build an extractive question-answering system, we need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A reader model to extract answers

We will use the SQuAD dataset, which consists of **questions** and **context** paragraphs containing question **answers**. We generate embeddings for the context passages using the retriever, index them in the vector database, and query with semantic search to retrieve the top k most relevant contexts containing potential answers to our question. We then use the reader model to extract the answers from the returned contexts.

Let's get started by installing the packages needed for notebook to run:

# Install Dependencies

In [1]:
!pip install -q \
    transformers==4.46.3 \
    datasets==4.5.0 \
    pinecone==5.4.2 \
    sentence-transformers==3.3.1 \
    huggingface_hub==0.26.5
    
!rm -rf ~/.cache/huggingface

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.26.5 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.26.5 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.10.0 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.26.5 which is incompatible.


In [1]:
# Force-kill the kernel process to restart it, ensuring the newly
# installed/uninstalled package versions are loaded in a fresh process

import os
import sys

print(
    "⚠️ WARNING:\n"
    "This code cell will restart the kernel to load the newly installed packages...\n"
    "If you used 'Run all', it will stop here — that's expected.\n"
    "See the note below on how to continue running the rest of the notebook.",
    flush=True,
)
sys.stdout.flush()

os.kill(os.getpid(), 9)

⚠️ WARNING:
This code cell will restart the kernel to load the newly installed packages...
If you used 'Run all', it will stop here — that's expected.
See the note below on how to continue running the rest of the notebook.


: 

: 

: 

<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

⚠️ **Do not use "Run all" for this whole notebook in one go.** The cell above force-restarts the kernel (`os.kill`) so the newly installed packages load cleanly. Colab's "Run all" cannot continue past a kernel restart — it will always stop right after that cell, no matter how many times you retry it.

Instead, run it in two steps:
1. Run only the two cells above (install + restart). Wait for the kernel to finish restarting (a few seconds).
2. Click on the next cell below, then use **Runtime → Run after** (or manually run the remaining cells one by one / select them and press Shift+Enter). Do not click "Run all" again, since it would just re-trigger the restart at the same spot.

</div>

# Load Dataset

Now let's load the SQUAD dataset from the HuggingFace Model Hub. We load the dataset into a pandas dataframe and filter the title, question, and context columns, and we drop any duplicate context passages.

In [21]:
from datasets import load_dataset

# load the squad dataset into a pandas dataframe
df = load_dataset("rajpurkar/squad", split="train").to_pandas()

In [22]:
# select only title and context column
df = df[["title", "context"]]
# drop rows containing duplicate context passages
df = df.drop_duplicates(subset="context")
df

,title,context
0,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha..."
5,University_of_Notre_Dame,"As at most other universities, Notre Dame's st..."
10,University_of_Notre_Dame,The university is the major seat of the Congre...
15,University_of_Notre_Dame,The College of Engineering was established in ...
20,University_of_Notre_Dame,All of Notre Dame's undergraduate students are...
...,...,...
87574,Kathmandu,"Institute of Medicine, the central college of ..."
87579,Kathmandu,Football and Cricket are the most popular spor...
87584,Kathmandu,The total length of roads in Nepal is recorded...
87589,Kathmandu,The main international airport serving Kathman...


# Pinecone Setup

Pinecone is a managed vector database that lets us store context embeddings and run fast semantic search over them. In this section, we set up the connection to Pinecone using an API key and create the index that will hold our context embeddings.

In [3]:
import os
import getpass
from pinecone import Pinecone

#
# For this notebook, you'll need a Pinecone API key, 
# to get one, sign up for free at https://app.pinecone.io/,
# then go to "API Keys" and copy the default key (or create a new one).
#
# Once you have your Pinecone API key,
# load it from Colab's "Secrets" manager 
# (the key icon in the left sidebar). 
# Add a secret named PINECONE_API_KEY there and enable notebook access.
#
# Note: Colab Secrets rely on a UI popup to grant access, which only works when
# running in the Colab UI itself. If you're running this notebook through
# another editor (e.g. VS Code's Colab extension), that popup isn't available,
# so we fall back to an environment variable or a secure manual prompt instead.
api_key = os.environ.get('PINECONE_API_KEY')


if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('PINECONE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    api_key = getpass.getpass('Enter your Pinecone API key: ')



<br>

Once we have the API key, we can initialize the connection:

In [23]:
from pinecone import Pinecone, ServerlessSpec

spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

# connect to pinecone environment
pc = Pinecone(
    api_key = api_key
)

Now we create a new index called "question-answering" — we can name the index anything we want. We specify the metric type as "cosine" and dimension as 384 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 384-dimension vectors.

In [24]:
index_name = "extractive-question-answering"

# check if the extractive-question-answering index exists
if index_name not in pc.list_indexes().names():
    # create the index if it does not exist
    pc.create_index(
        name=index_name,
        dimension=384,      # output size of multi-qa-MiniLM-L6-cos-v1
        metric="cosine",
        spec=spec,          # the ServerlessSpec defined above
    )
# connect to extractive-question-answering index we created
index = pc.Index(index_name)

# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all context passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will generate embeddings in a way that the questions and context passages containing answers to our questions are nearby in the vector space. We can use cosine similarity to calculate the similarity between the query and context embeddings to find the context passages that contain potential answers to our question.

We will use a SentenceTransformer model named ``multi-qa-MiniLM-L6-cos-v1`` designed for semantic search and trained on 215M (question, answer) pairs from diverse sources as our retriever.

In [25]:
import torch
from sentence_transformers import SentenceTransformer

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# load the retriever model from huggingface model hub
retriever = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1', device=device)

retriever

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, context passage, etc.

In [26]:
from tqdm.auto import tqdm

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(df), batch_size)):
    # find end of batch
    i_end = min(i + batch_size, len(df))
    # extract batch
    batch = df.iloc[i:i_end]
    # generate embeddings for batch
    emb = retriever.encode(batch["context"].tolist()).tolist()
    # get metadata
    meta = batch.to_dict(orient="records")
    # create unique IDs
    ids = [str(idx) for idx in batch.index]
    # add all to upsert list
    to_upsert = list(zip(ids, emb, meta))
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/296 [00:00<?, ?it/s]

{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 18891}},
 'total_vector_count': 18891}

# Initialize Reader

We use the `deepset/electra-base-squad2` model from the HuggingFace model hub as our reader model. We load this model into a "question-answering" pipeline from HuggingFace transformers and feed it our questions and context passages individually. The model gives a prediction for each context we pass through the pipeline.

In [27]:
from transformers import pipeline

model_name = 'deepset/electra-base-squad2'
# load the reader model into a question-answering pipeline

device_id = 0 if torch.cuda.is_available() else -1
reader = pipeline(tokenizer=model_name, model=model_name, task='question-answering', device=device_id)

reader

config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Now all the components we need are ready. Let's write some helper functions to execute our queries. The `get_context` function retrieves the context embeddings containing answers to our question from the Pinecone index, and the `extract_answer` function extracts the answers from these context passages.

In [29]:
# gets context passages from the pinecone index
def get_context(question, top_k):
    # generate embeddings for the question
    xq = retriever.encode(question).tolist()
    # search pinecone index for context passage with the answer
    xc = index.query(vector=xq, top_k=top_k, include_metadata=True)
    # extract the context passage from pinecone search result
    c = [match["metadata"]["context"] for match in xc["matches"]]
    return c



In [30]:
from pprint import pprint

# extracts answer from the context passage
def extract_answer(question, context):
    results = []
    for c in context:
        # feed the reader the question and contexts to extract answers
        answer = reader(question=question, context=c)
        # add the context to answer dict for printing both together
        answer["context"] = c
        results.append(answer)
    # sort the result based on the score from reader model
    sorted_result = pprint(sorted(results, key=lambda x: x['score'], reverse=True))
    return sorted_result

In [31]:
question = "How much oil is Egypt producing in a day?"
context = get_context(question, top_k = 1)
context

['Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.']

As we can see, the retiever is working fine and gets us the context passage that contains the answer to our question. Now let's use the reader to extract the exact answer from the context passage.

In [32]:
extract_answer(question, context)

[{'answer': '691,000 bbl/d',
  'context': 'Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of '
             'natural gas (in 2013), which makes Egypt as the largest oil '
             'producer not member of the Organization of the Petroleum '
             'Exporting Countries (OPEC) and the second-largest dry natural '
             'gas producer in Africa. In 2013, Egypt was the largest consumer '
             'of oil and natural gas in Africa, as more than 20% of total oil '
             'consumption and more than 40% of total dry natural gas '
             'consumption in Africa. Also, Egypt possesses the largest oil '
             'refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is '
             'currently planning to build its first nuclear power plant in El '
             'Dabaa city, northern Egypt.',
  'end': 33,
  'score': 0.9999852180480957,
  'start': 20}]


The reader model predicted with 99% accuracy the correct answer *691,000 bbl/d* as seen from the context passage. Let's run few more queries.

In [33]:
question = "What are the first names of the men that invented youtube?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'answer': 'Hurley and Chen',
  'context': 'According to a story that has often been repeated in the media, '
             'Hurley and Chen developed the idea for YouTube during the early '
             'months of 2005, after they had experienced difficulty sharing '
             "videos that had been shot at a dinner party at Chen's apartment "
             'in San Francisco. Karim did not attend the party and denied that '
             'it had occurred, but Chen commented that the idea that YouTube '
             'was founded after a dinner party "was probably very strengthened '
             'by marketing ideas around creating a story that was very '
             'digestible".',
  'end': 79,
  'score': 0.9999276399612427,
  'start': 64}]


In [34]:
question = "What is Albert Eistein famous for?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'answer': 'his theories of special relativity and general relativity',
  'context': 'Albert Einstein is known for his theories of special relativity '
             'and general relativity. He also made important contributions to '
             'statistical mechanics, especially his mathematical treatment of '
             'Brownian motion, his resolution of the paradox of specific '
             'heats, and his connection of fluctuations and dissipation. '
             'Despite his reservations about its interpretation, Einstein also '
             'made contributions to quantum mechanics and, indirectly, quantum '
             'field theory, primarily through his theoretical studies of the '
             'photon.',
  'end': 86,
  'score': 0.9500371217727661,
  'start': 29}]


Let's run another question. This time for top 3 context passages from the retriever.

In [35]:
question = "Who was the first person to step foot on the moon?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': 'Armstrong',
  'context': 'The trip to the Moon took just over three days. After achieving '
             'orbit, Armstrong and Aldrin transferred into the Lunar Module, '
             'named Eagle, and after a landing gear inspection by Collins '
             'remaining in the Command/Service Module Columbia, began their '
             'descent. After overcoming several computer overload alarms '
             'caused by an antenna switch left in the wrong position, and a '
             'slight downrange error, Armstrong took over manual flight '
             'control at about 180 meters (590 ft), and guided the Lunar '
             'Module to a safe landing spot at 20:18:04 UTC, July 20, 1969 '
             '(3:17:04 pm CDT). The first humans on the Moon would wait '
             'another six hours before they ventured out of their craft. At '
             '02:56 UTC, July 21 (9:56 pm CDT July 20), Armstrong became the '
             'first human to set foot on the Moon.',

<br>

### 🚀 Challenge

Try adding a few more questions of your own in new cells below (reusing `get_context` and `extract_answer`). Experiment with different `top_k` values and observe:

- Does the retriever always return a context that actually contains the answer?
- How does the reader's confidence score change with harder or more ambiguous questions?
- What happens with questions that have no good answer in the dataset?

### Challenge helper

`extract_answer` prints whole passages, which makes scores hard to compare. `ask` prints one line per retrieved passage instead.

In [37]:
def ask(question, top_k=3, show_context=False, allow_no_answer=False):
    print(f"\nQ: {question}  (top_k={top_k})")
    contexts = get_context(question, top_k)
    for rank, c in enumerate(contexts, 1):
        a = reader(question=question, context=c,
                   handle_impossible_answer=allow_no_answer)
        answer = a["answer"] if a["answer"] else "<no answer>"
        print(f"  #{rank}  score={a['score']:.3f}  answer={answer!r}")
        if show_context:
            print(f"       context: {c[:150]}...")

### 1. Does the retriever find the right passage? (varying `top_k`)

Check whether the right passage is ranked #1, or only shows up at #3 or #5, and whether the reader's highest score lands on the correct passage.

In [38]:
question = "In what year did Chopin leave Poland?"
for k in [1, 3, 5]:
    ask(question, top_k=k, show_context=True)


Q: In what year did Chopin leave Poland?  (top_k=1)
  #1  score=0.992  answer='1830'
       context: Chopin's successes as a composer and performer opened the door to western Europe for him, and on 2 November 1830, he set out, in the words of Zdzisław...

Q: In what year did Chopin leave Poland?  (top_k=3)
  #1  score=0.992  answer='1830'
       context: Chopin's successes as a composer and performer opened the door to western Europe for him, and on 2 November 1830, he set out, in the words of Zdzisław...
  #2  score=0.990  answer='1849'
       context: Frédéric François Chopin (/ˈʃoʊpæn/; French pronunciation: ​[fʁe.de.ʁik fʁɑ̃.swa ʃɔ.pɛ̃]; 22 February or 1 March 1810 – 17 October 1849), born Frydery...
  #3  score=1.000  answer='1848'
       context: Chopin made his last public appearance on a concert platform at London's Guildhall on 16 November 1848, when, in a final patriotic gesture, he played ...

Q: In what year did Chopin leave Poland?  (top_k=5)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  #1  score=0.992  answer='1830'
       context: Chopin's successes as a composer and performer opened the door to western Europe for him, and on 2 November 1830, he set out, in the words of Zdzisław...
  #2  score=0.990  answer='1849'
       context: Frédéric François Chopin (/ˈʃoʊpæn/; French pronunciation: ​[fʁe.de.ʁik fʁɑ̃.swa ʃɔ.pɛ̃]; 22 February or 1 March 1810 – 17 October 1849), born Frydery...
  #3  score=1.000  answer='1848'
       context: Chopin made his last public appearance on a concert platform at London's Guildhall on 16 November 1848, when, in a final patriotic gesture, he played ...
  #4  score=0.999  answer='1848'
       context: In late summer he was invited by Jane Stirling to visit Scotland, where he stayed at Calder House near Edinburgh and at Johnstone Castle in Renfrewshi...
  #5  score=0.999  answer='1831'
       context: Chopin arrived in Paris in late September 1831; he would never return to Poland, thus becoming one of many expatriates of the Polish Great 

### 2. Easy vs. harder or ambiguous questions

Specific questions usually score high. Ambiguous ones tend to pull passages from unrelated articles, with lower, less consistent scores.

In [39]:
questions = [
    "How much oil is Egypt producing in a day?",          # specific, wording close to the text
    "What is the Grotto at Notre Dame?",                   # specific
    "Who was the president?",                              # ambiguous: president of what?
    "When did it start?",                                  # no subject at all
    "Which instrument did the Polish composer mainly write for?",  # paraphrased, no names
]
for q in questions:
    ask(q, top_k=3)


Q: How much oil is Egypt producing in a day?  (top_k=3)
  #1  score=1.000  answer='691,000 bbl/d'
  #2  score=0.000  answer='600,000 tonnes'
  #3  score=0.000  answer='six million barrels per day'

Q: What is the Grotto at Notre Dame?  (top_k=3)
  #1  score=1.000  answer='a Marian place of prayer and reflection'
  #2  score=0.992  answer='a replica of the original in Lourdes, France'
  #3  score=0.000  answer='Our Lady of the Lake'

Q: Who was the president?  (top_k=3)
  #1  score=0.273  answer='Mamadu Ture Kuruma'
  #2  score=0.341  answer='Franklin Roosevelt'
  #3  score=0.999  answer='Boris Yeltsin'

Q: When did it start?  (top_k=3)
  #1  score=0.997  answer='30 April 1916'
  #2  score=0.520  answer='30 October 2000, followed by BeiDou-1B on 20 December 2000'
  #3  score=0.619  answer='1245'

Q: Which instrument did the Polish composer mainly write for?  (top_k=3)
  #1  score=0.816  answer='virtuosi'
  #2  score=1.000  answer='solo piano'
  #3  score=0.826  answer='piano'


### 3. Questions with no answer in the dataset

By default the pipeline always returns some text span, even when no passage answers the question. The reader was trained on SQuAD 2.0, so with `allow_no_answer=True` it can return `<no answer>` instead. Compare the two runs.

In [40]:
no_answer_questions = [
    "Who won the 2022 FIFA World Cup?",   # SQuAD comes from Wikipedia in 2016
    "What is the capital of Atlantis?",
    "What is the airspeed velocity of an unladen swallow?",
]
for q in no_answer_questions:
    ask(q, top_k=3)                        # default: reader must pick some span
    ask(q, top_k=3, allow_no_answer=True)  # let the reader return "no answer"


Q: Who won the 2022 FIFA World Cup?  (top_k=3)
  #1  score=0.000  answer='The England team won their first and only World Cup title in 1966'
  #2  score=0.000  answer='Manchester United in 2008)'
  #3  score=0.000  answer='Josep Guardiola'

Q: Who won the 2022 FIFA World Cup?  (top_k=3)
  #1  score=1.000  answer='<no answer>'
  #2  score=0.990  answer='<no answer>'
  #3  score=1.000  answer='<no answer>'

Q: What is the capital of Atlantis?  (top_k=3)
  #1  score=0.000  answer='MGM Grand Atlantic City'
  #2  score=0.000  answer='Neapolis'
  #3  score=0.000  answer='Leulumoega'

Q: What is the capital of Atlantis?  (top_k=3)
  #1  score=1.000  answer='<no answer>'
  #2  score=0.991  answer='<no answer>'
  #3  score=1.000  answer='<no answer>'

Q: What is the airspeed velocity of an unladen swallow?  (top_k=3)
  #1  score=0.000  answer='1 m (3 ft) per day'
  #2  score=0.000  answer='water'
  #3  score=0.000  answer='below 600 m'

Q: What is the airspeed velocity of an unladen swallow?  

### Observations

**Does the retriever always return a context that contains the answer?**
No. For specific questions that reuse words from the text (e.g. the Egypt oil and Chopin questions), the correct passage is usually ranked #1. For paraphrased or vague questions, the right passage sometimes only appears at rank 3 or 5, or not at all. Increasing `top_k` raises the chance that the answer is somewhere in the retrieved set, but it also hands the reader more irrelevant passages to choose from. The retriever finds passages that are *semantically similar* to the question, which is not the same as passages that *contain the answer*.

**How does the reader's confidence change with harder or more ambiguous questions?**
For specific, well-phrased questions, the reader's top score is high (often above 0.9) and clearly separated from the other passages. For ambiguous questions such as "Who was the president?" or "When did it start?", the scores are lower and closer together, and the "answers" come from unrelated articles. The reader still always picks a span, so a plausible-looking answer doesn't mean it's correct. The score is a useful signal, but not a guarantee.

**What happens with questions that have no good answer in the dataset?**
By default the pipeline always returns *some* span. For questions outside SQuAD's 2016 Wikipedia content (e.g. the 2022 World Cup), it extracts an unrelated name or date, sometimes with a deceptively moderate score. With `handle_impossible_answer=True`, the SQuAD 2.0–trained reader can return no answer instead, which gives more honest results for these questions. A real system would need this option, or a score threshold, so that it doesn't present confident-sounding wrong answers.

**Overall:** the pipeline is only as good as its weakest stage. Retrieval errors (wrong passage) can't be fixed by the reader, and the reader will make up a span rather than admit that the answer isn't there, unless you explicitly allow it to.

<br>

---

### Clean Up

Now that we're done experimenting, let's delete the Pinecone index we created. This frees up the resources it was using (and, when using a paid plan, avoids being billed for an index we no longer need).

In [ ]:
pc.delete_index(index_name)